<a href="https://colab.research.google.com/github/fabriciosantana/mcdia/blob/main/10-adap/assignments/02/atividade_qualidade_fornecedores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instituto Brasileiro de Ensino, Desenvolvimento e Pesquisa**

Programa de Pós-Graduação em Administração Pública — Mestrado Profissional

**Disciplina:** Auditoria de dados e accountability com python

**Professor:** Aloísio Dourado Neto

**Atividade:** Exercício do Módulo IV - Avaliação da qualidade dos dados cadastrais de fornecedores

**Grupo:** Fabricio Santana e Giovanni Brígido

# Atividade 2 — Qualidade dos dados cadastrais de fornecedores

Este notebook avalia completude, unicidade, consistência e similaridade dos cadastros. Coloque nesta pasta os arquivos `fornecedores_transf.pkl`, `municipios_transf.pkl`, `cnaes_transf.pkl` e `CNAERFB.CSV` antes de executar as células.

In [ ]:
from pathlib import Path
import csv
import unicodedata

import pandas as pd


def localizar(nome):
    """Localiza um arquivo na pasta de execução e em seus diretórios ascendentes."""
    candidatos = [Path.cwd() / nome, Path(__file__).resolve().parent / nome] if "__file__" in globals() else [Path.cwd() / nome]
    for origem in list(candidatos):
        candidatos.extend(origem.parent.parent.glob(nome))
    for caminho in candidatos:
        if caminho.is_file():
            return caminho
    raise FileNotFoundError(f"Arquivo não encontrado: {nome}. Coloque-o em 10-adap/assignments/02.")

def sem_acentos(valor):
    valor = unicodedata.normalize("NFKD", str(valor))
    return "".join(c for c in valor if not unicodedata.combining(c))

def coluna(df, *nomes):
    normalizadas = {sem_acentos(c).strip().lower().replace(" ", "_"): c for c in df.columns}
    for nome in nomes:
        chave = sem_acentos(nome).strip().lower().replace(" ", "_")
        if chave in normalizadas:
            return normalizadas[chave]
    raise KeyError(f"Nenhuma destas colunas foi encontrada: {nomes}. Colunas: {list(df.columns)}")

def preenchido(serie):
    return serie.notna() & serie.astype("string").str.strip().ne("") & serie.astype("string").str.lower().ne("nan")

fornecedores_path = localizar("fornecedores_transf.pkl")
municipios_path = localizar("municipios_transf.pkl")
cnaes_path = localizar("cnaes_transf.pkl")
cnae_rfb_path = localizar("CNAERFB.CSV")

fornecedores = pd.read_pickle(fornecedores_path)
municipios = pd.read_pickle(municipios_path)
cnaes = pd.read_pickle(cnaes_path)

fornecedores.shape, municipios.shape, cnaes.shape

## Completude

Cada fornecedor deve possuir pelo menos um documento de identificação preenchido: CNPJ ou CPF.

In [ ]:
cnpj = coluna(fornecedores, "CNPJ", "cnpj")
cpf = coluna(fornecedores, "CPF", "cpf")

fornecedores["tem_cnpj"] = preenchido(fornecedores[cnpj])
fornecedores["tem_cpf"] = preenchido(fornecedores[cpf])
sem_documento = fornecedores.loc[~(fornecedores["tem_cnpj"] | fornecedores["tem_cpf"])].copy()

completude = pd.DataFrame({
    "fornecedores": [len(fornecedores)],
    "sem_documento": [len(sem_documento)],
    "percentual_com_documento": [100 * (1 - len(sem_documento) / len(fornecedores)) if len(fornecedores) else 0],
})
completude

In [ ]:
sem_documento

## Unicidade

São identificados CNPJs repetidos e cadastros que possuem CNPJ e CPF simultaneamente.

In [ ]:
cnpj_preenchido = fornecedores.loc[fornecedores["tem_cnpj"]].copy()
cnpj_duplicados = cnpj_preenchido[cnpj_preenchido[cnpj].duplicated(keep=False)].sort_values(cnpj)
cnpj_e_cpf = fornecedores.loc[fornecedores["tem_cnpj"] & fornecedores["tem_cpf"]].copy()

pd.DataFrame({
    "cnpjs_duplicados": [cnpj_duplicados[cnpj].nunique()],
    "registros_com_cnpj_duplicado": [len(cnpj_duplicados)],
    "registros_com_cnpj_e_cpf": [len(cnpj_e_cpf)],
})

In [ ]:
cnpj_duplicados

In [ ]:
cnpj_e_cpf

## Consistência

As UFs dos fornecedores são comparadas com as UFs existentes no cadastro de municípios.

In [ ]:
uf_fornecedores = coluna(fornecedores, "UF", "uf")
uf_municipios = coluna(municipios, "UF", "uf")
ufs_fornecedores = set(fornecedores.loc[preenchido(fornecedores[uf_fornecedores]), uf_fornecedores].astype("string").str.strip().str.upper())
ufs_municipios = set(municipios.loc[preenchido(municipios[uf_municipios]), uf_municipios].astype("string").str.strip().str.upper())
ufs_sem_correspondencia = sorted(ufs_fornecedores - ufs_municipios)

consistencia = pd.DataFrame({
    "uf_fornecedores": [sorted(ufs_fornecedores)],
    "uf_municipios": [sorted(ufs_municipios)],
    "ufs_sem_correspondencia": [ufs_sem_correspondencia],
    "todas_correspondem": [not ufs_sem_correspondencia],
})
consistencia

## Similaridade dos códigos CNAE

Os códigos em comum são comparados após padronização textual. Em caso de descrições distintas, é calculada a distância de Levenshtein.

In [ ]:
def padronizar_codigo(serie):
    return (serie.astype("string").str.replace(r"\D", "", regex=True).str.lstrip("0").replace("", "0"))

def ler_cnae_rfb(caminho):
    for encoding in ("utf-8-sig", "latin1"):
        try:
            return pd.read_csv(caminho, sep=None, engine="python", encoding=encoding, dtype="string")
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("csv", b"", 0, 1, "encoding não suportado")

cnae_rfb = ler_cnae_rfb(cnae_rfb_path)
codigo_cnaes = coluna(cnaes, "codigo", "código", "codigo_cnae", "cnae")
descricao_cnaes = coluna(cnaes, "descricao", "descrição", "descricao_cnae")
codigo_rfb = coluna(cnae_rfb, "codigo", "código", "codigo_cnae", "cnae")
descricao_rfb = coluna(cnae_rfb, "descricao", "descrição", "descricao_cnae")

cnaes_comparacao = cnaes[[codigo_cnaes, descricao_cnaes]].rename(columns={codigo_cnaes: "codigo", descricao_cnaes: "descricao_cnaes"}).copy()
rfb_comparacao = cnae_rfb[[codigo_rfb, descricao_rfb]].rename(columns={codigo_rfb: "codigo", descricao_rfb: "descricao_rfb"}).copy()
for frame in (cnaes_comparacao, rfb_comparacao):
    frame["codigo"] = padronizar_codigo(frame["codigo"])
    frame[frame.columns[-1]] = frame[frame.columns[-1]].astype("string").str.strip()

comparacao = cnaes_comparacao.merge(rfb_comparacao, on="codigo", how="inner")
descricoes_diferentes = comparacao.loc[
    comparacao["descricao_cnaes"].fillna("").str.casefold() != comparacao["descricao_rfb"].fillna("").str.casefold()
].copy()
len(comparacao), len(descricoes_diferentes)

In [ ]:
def distancia_levenshtein(a, b):
    a, b = str(a), str(b)
    if len(a) < len(b):
        a, b = b, a
    linha = list(range(len(b) + 1))
    for i, caractere_a in enumerate(a, 1):
        nova = [i]
        for j, caractere_b in enumerate(b, 1):
            nova.append(min(nova[-1] + 1, linha[j] + 1, linha[j - 1] + (caractere_a != caractere_b)))
        linha = nova
    return linha[-1]

descricoes_diferentes["distancia_levenshtein"] = descricoes_diferentes.apply(
    lambda linha: distancia_levenshtein(linha["descricao_cnaes"], linha["descricao_rfb"]), axis=1
)
descricoes_diferentes.sort_values(["distancia_levenshtein", "codigo"])[["codigo", "descricao_cnaes", "descricao_rfb", "distancia_levenshtein"]]

## Conclusão

As tabelas e indicadores acima registram os achados de cada dimensão. A coluna `todas_correspondem` informa diretamente se todas as UFs dos fornecedores estão presentes no cadastro de municípios; `descricoes_diferentes` detalha os códigos CNAE com divergência e sua distância de Levenshtein.